In [9]:
import os
import time
import pandas as pd
import yfinance as yf
import kagglehub

# PATHS
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.join(REPO_ROOT, "data", "market")
KAGGLE_DATASET = "andrewmvd/sp-500-stocks"
PRICES_CACHE = os.path.join(DATA_DIR, "sp500_prices.csv")
MAX_AGE_DAYS = 30

def download_kaggle_dataset():
    os.makedirs(DATA_DIR, exist_ok=True)
    kagglehub.dataset_download(KAGGLE_DATASET, output_dir=DATA_DIR)


def load_sp500_companies_index_df():
    """Read CSV files, load sp500 companies name(ticker) and sp500 index by dates

    Returns:
        _type_: _description_
    """
    res = {}
    for value in ["companies", "index"]:
        df = pd.read_csv(os.path.join(DATA_DIR, f"sp500_{value}.csv"))
        res[value] = df

    return res


# Kaggle Data
def load_kaggle_data(force_update=False):
    """load_kaggle_data : Load a dataset for sp500 companies name and sp500 index
    Use Cached dataset if not too old, else download new dataset from Kaggle

    Args:
        force_update (bool, optional): _description_. Defaults to False.

    Returns:
        _type_: _description_
    """
    companies_path = os.path.join(DATA_DIR, "sp500_companies.csv")
    if not force_update and os.path.exists(companies_path):
        cached_data_age = (time.time() - os.path.getmtime(companies_path)) / 86400
        if cached_data_age < MAX_AGE_DAYS:
            # Use Cached Dataset
            print(f"Using cached Kaggle data ({cached_data_age:.0f} days old).")
        else:
            # Cache outdated, new dataset
            print(f"Kaggle cache is {cached_data_age:.0f} days old, re-downloading...")
            download_kaggle_dataset()
    else:
        # Force update or first run (file doesn't exist yet)
        print("Downloading Kaggle dataset...")
        download_kaggle_dataset()

    return load_sp500_companies_index_df()


# Price Data (cached)



def clean_raw_price_data(raw_prices_df):
    raw_prices_df = raw_prices_df.dropna(axis=1, thresh=int(0.95 * len(raw_prices_df)))
    raw_prices_df = raw_prices_df.ffill().dropna()
    raw_prices_df.index.name = "Date"
    raw_prices_df.to_csv(PRICES_CACHE)
    print(
        f"Saved {raw_prices_df.shape[1]} stocks x {raw_prices_df.shape[0]} days → {PRICES_CACHE}"
    )
    return raw_prices_df


def download_yf_prices_data(tickers, start, end):
    print("Downloading prices from yfinance (this takes ~2 min)...")
    os.makedirs(DATA_DIR, exist_ok=True)
    raw = yf.download(tickers, start=start, end=end, auto_adjust=False)[
        "Adj Close"
    ]  # Use 'Adj Close'

    # raw = raw.dropna(axis=1, thresh=int(0.95 * len(raw)))
    # raw = raw.ffill().dropna()
    # raw.index.name = "Date"
    # raw.to_csv(PRICES_CACHE)
    # print(f"Saved {raw.shape[1]} stocks x {raw.shape[0]} days → {PRICES_CACHE}")
    return clean_raw_price_data(raw_prices_df=raw)

def check_cached_prices_data(force_update):
    """Return cached DataFrame if fresh, else None (caller must re-download)."""
    if not force_update and os.path.exists(PRICES_CACHE):
        age = (time.time() - os.path.getmtime(PRICES_CACHE)) / 86400
        if age < MAX_AGE_DAYS:
            print(f"Loading cached prices ({age:.0f} days old)...")
            return pd.read_csv(PRICES_CACHE, index_col="Date", parse_dates=True)
        print(f"Price cache is {age:.0f} days old, will re-download.")
        return None
    return None

def load_prices(tickers, start="2015-01-01", end="2024-12-31", force_update=False):
    """load_prices Load Prices data using cached data / yfinacne

    Args:
        tickers (_type_): _description_
        start (str, optional): _description_. Defaults to "2015-01-01".
        end (str, optional): _description_. Defaults to "2024-12-31".
        force_update (bool, optional): _description_. Defaults to False.

    Returns:
        _type_: _description_
    """
    cached_data = check_cached_prices_data(force_update)

    if cached_data is not None:
        return cached_data
    else:
        return download_yf_prices_data(tickers, start, end)


# Usage
dfs = load_kaggle_data()
tickers = dfs["companies"]["Symbol"].str.replace(".", "-", regex=False).tolist()

# ----------------------------

print("Downloading prices from yfinance (this takes ~2 min)...")
os.makedirs(DATA_DIR, exist_ok=True)
raw = yf.download(tickers, start="2025-12-31", end="2026-01-31", auto_adjust=False)[
    "Adj Close"
]



# Load Prices
# prices = load_prices(tickers)
# dfs = load_kaggle_data()

# tickers = dfs["companies"]["Symbol"].str.replace(".", "-", regex=False).tolist()

# # 2. Load prices
# prices = load_prices(tickers)
# stock_names = prices.columns.tolist()
# print(f"Prices shape: {prices.shape}  (days x stocks)")



[                       1%                       ]  3 of 502 completed

Using cached Kaggle data (12 days old).


[**********************95%*********************  ]  477 of 502 completedFailed to get ticker 'MMC' reason: Failed to perform, curl: (7) Failed to connect to query1.finance.yahoo.com port 443 after 2002 ms: Could not connect to server. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[*********************100%***********************]  502 of 502 completed

18 Failed downloads:
['WBA', 'HES', 'ANSS', 'FI', 'DAY', 'IPG', 'PARA', 'K', 'JNPR', 'DFS', 'MMC']: YFTzMissingError('possibly delisted; no timezone found')
['HD']: ConnectionError('Failed to perform, curl: (7) Failed to connect to query2.finance.yahoo.com port 443 after 2001 ms: Could not connect to server. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['MPWR', 'EXR', 'FTV', 'CCL']: ConnectionError('Failed to perform, curl: (7) Failed to connect to query2.finance.yahoo.com port 443 after 2002 ms: Could not connect to server. See https://curl.se/libcurl/c/libcurl-errors.html first

In [12]:
raw

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-31,135.518021,271.355835,224.801315,135.720001,123.881355,95.919998,266.043854,349.989990,270.353485,56.686184,...,327.511169,23.486528,119.753548,73.318069,118.737274,135.189789,149.852646,89.676476,242.820007,124.742447
2026-01-02,137.390381,270.507446,225.608078,133.009995,122.793724,93.860001,257.764099,333.299988,272.885559,58.224369,...,325.188873,23.595581,121.982819,74.132050,121.016518,136.113037,149.070114,89.776207,248.300003,124.841591
2026-01-05,142.350174,266.764404,216.625458,135.869995,125.028305,94.440002,261.760193,331.559998,276.424469,58.746960,...,333.491302,24.051632,119.106659,73.526527,123.690422,137.989304,148.871979,92.399086,252.639999,128.262039
2026-01-06,147.263977,261.873474,220.314926,138.020004,126.343353,93.919998,273.609680,335.989990,292.025635,59.644238,...,334.986328,23.783951,119.952583,73.883888,119.437820,140.302353,149.981415,94.493401,257.609985,128.242218
2026-01-07,147.473495,259.847229,229.651733,137.039993,125.611679,95.160004,271.676117,338.100006,291.975800,57.701778,...,330.281952,22.980907,115.812523,72.682770,116.911911,138.416153,149.070114,92.419029,246.960007,124.593735
2026-01-08,145.428162,258.559631,220.511703,138.660004,124.761345,96.379997,279.450165,339.040009,298.226227,60.334450,...,332.036133,23.783951,117.703407,72.841591,121.273056,139.865555,152.031891,93.825218,254.639999,125.763626
2026-01-09,148.181885,258.889008,216.527084,139.270004,124.504265,95.180000,278.309845,333.950012,299.990692,61.073959,...,329.165680,25.509005,117.265518,73.715134,122.950409,138.674286,154.052643,92.598541,259.480011,126.080887
2026-01-12,147.653091,259.767365,216.487717,138.509995,123.238663,93.239998,278.597382,327.649994,292.942749,61.330330,...,328.358368,25.528833,116.280258,73.457039,122.378128,138.068710,155.439423,91.202332,264.429993,124.246735
2026-01-13,145.986893,260.565887,217.186279,140.070007,122.902481,90.769997,274.472382,309.929993,295.285400,62.493832,...,326.335083,25.231411,115.842377,74.390144,124.854698,139.091217,154.884720,88.100754,261.440002,123.552727


In [ ]:
raw = raw.dropna(axis=1, thresh=int(0.95 * len(raw)))   # Keep columns where 95% of values are filled


In [11]:
raw = raw.dropna(axis=1, thresh=int(0.95 * len(raw)))
raw = raw.ffill().dropna()
raw.index.name = "Date"
raw.to_csv(PRICES_CACHE)
print(
    f"Saved {raw.shape[1]} stocks x {raw.shape[0]} days → {PRICES_CACHE}"
)


Saved 484 stocks x 21 days → /Users/nautilus/gridfw/data/market/sp500_prices.csv
